# Lab 04 solution: a fourth tool, and our answers to the six jobs

The worked answers to the challenge of `04_what_is_an_agent.ipynb`, our answers to the two questions of
Checkpoint D, and our answers to Part G's six jobs, with the reasons. The setup repeats the notebook's lookups, schemas and agent, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
import json
from openai import OpenAI, omit

client = OpenAI(timeout=60)

The three lookups, their schemas and `desk_agent`, exactly as in the notebook's Part E.

In [3]:
import sqlite3

DB_PATH = ROOT / "data" / "chinook.db"
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True, check_same_thread=False)


def find_customer(name: str) -> list[dict]:
    """Find a shop customer by full name. Returns a list of matches with their customer_id.

    Args:
        name: First and last name, for example 'Ana Silva'.
    """
    rows = conn.execute(
        "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
        "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = conn.execute(
        "SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id: int) -> dict:
    """Name the support rep who looks after a customer."""
    row = conn.execute(
        "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
        "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "support_rep": row[0] if row else None}

In [4]:
TOOLS = [
    {"type": "function", "function": {
        "name": "find_customer",
        "description": "Find a shop customer by full name. Returns a list of matches with their customer_id.",
        "parameters": {"type": "object",
                       "properties": {"name": {"type": "string",
                                               "description": "First and last name, for example 'Ana Silva'."}},
                       "required": ["name"], "additionalProperties": False},
        "strict": True}},
    {"type": "function", "function": {
        "name": "get_invoices",
        "description": "Count a customer's invoices and add up their total in US dollars.",
        "parameters": {"type": "object", "properties": {"customer_id": {"type": "integer"}},
                       "required": ["customer_id"], "additionalProperties": False},
        "strict": True}},
    {"type": "function", "function": {
        "name": "get_support_rep",
        "description": "Name the support rep who looks after a customer.",
        "parameters": {"type": "object", "properties": {"customer_id": {"type": "integer"}},
                       "required": ["customer_id"], "additionalProperties": False},
        "strict": True}},
]

FUNCTIONS = {
    "find_customer": find_customer,
    "get_invoices": get_invoices,
    "get_support_rep": get_support_rep,
}

print([tool["function"]["name"] for tool in TOOLS])

['find_customer', 'get_invoices', 'get_support_rep']


In [5]:
DESK_AGENT = (
    "You help the support reps of the Chinook music shop. Look up every fact with the tools; "
    "never answer from memory. Answer in one or two sentences."
)


def desk_agent(question, max_turns=6):
    messages = [
        {"role": "system", "content": DESK_AGENT},
        {"role": "user", "content": question},
    ]
    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, reasoning_effort=EFFORT or omit
        )
        reply = response.choices[0].message
        messages.append(reply)

        # Exit 1: the model answered
        if not reply.tool_calls:
            print(f"turn {turn}: the model answers")
            return reply.content

        # The model asked for lookups: our code runs the function each name points to
        for call in reply.tool_calls:
            print(f"turn {turn}: the model asks for {call.function.name}({call.function.arguments})")
            result = FUNCTIONS[call.function.name](**json.loads(call.function.arguments))
            content = json.dumps(result, ensure_ascii=False)
            print(f"        our code returns {content}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": content})

    # Exit 2: the turn cap stopped the loop, and there is no answer
    return None

## Task 1: ask the agent as it is

In [6]:
LATEST = "When was Helena Holý's latest invoice, and how much was it?"

print(desk_agent(LATEST))

turn 1: the model asks for find_customer({"name":"Helena Holý"})
        our code returns [{"customer_id": 6, "name": "Helena Holý", "country": "Czech Republic"}]


turn 2: the model asks for get_invoices({"customer_id":6})
        our code returns {"customer_id": 6, "invoice_count": 7, "total": 49.62}


turn 3: the model answers
I can’t determine the date or amount of Helena Holý’s latest invoice from the available invoice details; they only show she has 7 invoices totaling $49.62.


Three turns: `find_customer`, then `get_invoices`, then a reply that the date and amount of Helena's latest
invoice cannot be found in what the lookups returned; it gives the two facts it did get, 7 invoices and \$49.62. It
did not invent a date. None of the three tools returns one, and the reply says so.

## Task 2: add `get_latest_invoice`

The function first, checked by hand on Helena. `ORDER BY InvoiceDate DESC` puts the newest invoice first and
`LIMIT 1` keeps only that one; `date(InvoiceDate)` drops the time of day, which is always midnight in this database.

In [7]:
def get_latest_invoice(customer_id: int) -> dict:
    """Find a customer's most recent invoice: its id, its date and its total in US dollars."""
    row = conn.execute(
        "SELECT InvoiceId, date(InvoiceDate), Total FROM invoices WHERE CustomerId = ? "
        "ORDER BY InvoiceDate DESC LIMIT 1", (customer_id,)).fetchone()
    if row is None:
        return {"customer_id": customer_id, "latest_invoice": None}
    return {"customer_id": customer_id, "invoice_id": row[0], "date": row[1], "total": row[2]}


print(get_latest_invoice(6))

{'customer_id': 6, 'invoice_id': 404, 'date': '2013-11-13', 'total': 25.86}


Invoice 404, 2013-11-13, \$25.86: the values the challenge expects. Now its schema goes into `TOOLS` and the
function into `FUNCTIONS`. `desk_agent` reads both names each time it runs, so nothing else changes.

In [8]:
TOOLS.append({"type": "function", "function": {
    "name": "get_latest_invoice",
    "description": "Find a customer's most recent invoice: its id, its date and its total in US dollars.",
    "parameters": {"type": "object", "properties": {"customer_id": {"type": "integer"}},
                   "required": ["customer_id"], "additionalProperties": False},
    "strict": True}})
FUNCTIONS["get_latest_invoice"] = get_latest_invoice

print([tool["function"]["name"] for tool in TOOLS])
print(desk_agent(LATEST))

['find_customer', 'get_invoices', 'get_support_rep', 'get_latest_invoice']


turn 1: the model asks for find_customer({"name":"Helena Holý"})
        our code returns [{"customer_id": 6, "name": "Helena Holý", "country": "Czech Republic"}]


turn 2: the model asks for get_latest_invoice({"customer_id":6})
        our code returns {"customer_id": 6, "invoice_id": 404, "date": "2013-11-13", "total": 25.86}


turn 3: the model answers
Helena Holý’s latest invoice was dated November 13, 2013, for $25.86.


`TOOLS` now has four names. The model asked for `find_customer`, then for `get_latest_invoice` and nothing
else, and answered: *Helena Holý’s latest invoice was dated November 13, 2013, for \$25.86.* That is invoice 404's
date and total, as our hand check showed. We never told the model when to use the new tool: it read the tool's
description and chose it for this question.

## Task 3: what the workflow would need

`customer_workflow` has no step that reads a date. To answer this question we would write a new step (or a new
function, `latest_invoice_workflow`) and change the code that decides which function a question goes to. The agent
needed only one more tool: the model chose when to use it. That is the difference between the two shapes, seen from
the side of the person who maintains them.

## Part D: our answers to the checkpoint questions

- **To let the agent cancel an order, change the tools:** add a function such as `cancel_order(order_id)` and its
  schema. The model, the loop and the memory stay as they are; the instructions may gain a line on when to cancel.
- **Put a check in the guardrails, in our code, before the cancel runs:** for example, that the order exists, that
  it has not shipped, and that the person asking owns it, or that a person approves (Part G's draft level). The
  model writes the request; our code decides whether to run it.

## Part G: our answers to the six jobs

Rows 4 and 5 have more than one answer you could defend; these are ours, with the reason for each.

| # | Job | Shape | Who decides the next step? | Autonomy |
|---|---|---|---|---|
| 1 | Send the same overdue reminder to every account more than 30 days late | workflow | we do, in advance: the list and the message are known | act with audit |
| 2 | A visitor asks what jazz fusion is | chatbot | nobody: one reply | suggest |
| 3 | A messy email with an invoice question, an address change and a complaint | agent | the model: the next lookup depends on what it finds | draft |
| 4 | Refund duplicate charges straight from the inbox, up to a set limit, with no one in the loop | an agent could do it, but not with no one in the loop | the model would choose, and a wrong refund costs money | draft, or a rules workflow |
| 5 | Route each incoming request to billing, shipping or product | either | we do: one model call labels the request and our code picks the queue; the model, only for the requests that fit no label | act with audit on the clear ones, draft for the rest |
| 6 | Find out why revenue fell last quarter | agent | the model: the next query depends on the last result | suggest |

- **Row 1.** If you can write the steps down in advance, it is a workflow. The reminders are logged and can be
  resent or withdrawn, so act with audit. An agent here spends model calls to find an order of steps we already know.
- **Row 2.** The visitor needs only an explanation, so one reply does the job: there is nothing to look up and no
  loop to run.
- **Rows 3 and 6.** Nobody can write the path before seeing the email or the numbers, which is the case for an
  agent. Row 6 stays at suggest, because a wrong cause turns into a wrong business decision.
- **Row 4.** It looks like an agent's job, but it must not run at act: a wrong refund is paid before anyone sees it.
  A small rules workflow ("refund the later of two same-day invoices for the same album, up to the limit") or an
  agent at draft is the safer answer.
- **Row 5.** If the three categories are stable, a classifier and three queues is a workflow you can test. The label
  can come from one model call at a fixed step; our code still chooses what happens next, so it is a workflow
  (Part D). If many requests need two lookups before anyone knows the team, those are the agent's share.